In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

In [2]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, stratify=y_train, test_size=0.2, random_state=42, shuffle=True)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
# Just copy this syntax to a T
def create_example(image, label):
    image_example = Example(
        features=Features(
            feature={
                "image": Feature(bytes_list=BytesList(value=[tf.io.serialize_tensor(image).numpy()])),
                "label": Feature(int64_list=Int64List(value=[label]))
            }
        )
    )
    return image_example

print(create_example(X_train[0], y_train[0]))
features {
  feature {
    key: "image"
    value {
      bytes_list {
        value: "\010\004\022\..." // Thông tin của ảnh
      }
    }
  }
  feature {
    key: "label"
    value {
      int64_list {
        value: 6
      }
    }
  }
}

In [4]:
def write_tfrecords(f_name, X_data, y_data):
    with tf.io.TFRecordWriter(f_name) as file:
        for image, label in zip(X_data, y_data):
            ex = create_example(image.reshape(28, 28, 1),label)
            file.write(ex.SerializeToString())

In [15]:
write_tfrecords("data/train.tfrec", X_train, y_train)
write_tfrecords("data/validation.tfrec", X_valid, y_valid)
write_tfrecords("data/test.tfrec", X_test, y_test)

In [16]:
def parse_single_example(ser_ex): 
    feature_desc = {
        "image": tf.io.FixedLenFeature([], tf.string, default_value=""),
        "label": tf.io.FixedLenFeature([], tf.int64, default_value=-1)
    }
    parsed_ex = tf.io.parse_single_example(ser_ex, feature_desc)
    image = tf.io.parse_tensor(parsed_ex["image"], out_type=tf.uint8)
    image = tf.reshape(image, shape=[28, 28, 1])
    image = tf.cast(image, tf.float64)
    label = parsed_ex["label"]
    return image, label 

In [17]:
def get_dataset(filepath, n_read_threads=4, n_parse_threads=4, batch_size=32):
    dataset = tf.data.TFRecordDataset(filepath, num_parallel_reads=n_read_threads)
    dataset = dataset.map(parse_single_example, num_parallel_calls=n_parse_threads)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(1)

In [28]:
datasets = {
    "train": "data/train.tfrec",
    "validation": "data/validation.tfrec",
    "test": "data/test.tfrec"
}
train_dataset = get_dataset(datasets["train"])
validation_dataset = get_dataset(datasets["validation"])
test_dataset = get_dataset(datasets["test"])

In [19]:
class ImageStandardizeLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(ImageStandardizeLayer, self).__init__()
        
    def adapt(self, inputs):
        self.mean = tf.math.reduce_mean(inputs, axis=0, keepdims=True)
        self.std = tf.math.reduce_std(inputs, axis=0, keepdims=True)
        
    def call(self, inputs):
        results = tf.math.divide(tf.math.subtract(inputs, self.mean), (self.std + tf.keras.backend.epsilon()))
        return results

In [20]:
preprocess_layers = [
    ImageStandardizeLayer()
]

nn_layers = [
    tf.keras.layers.Conv2D(filters=64, kernel_size=5, strides=1, padding="same", activation="relu", input_shape=[28, 28, 1]),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(filters=128, kernel_size=3, strides=1, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(filters=128, kernel_size=3, strides=1, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(filters=256, kernel_size=3, strides=1, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(filters=256, kernel_size=3, strides=1, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(50, activation="relu", kernel_initializer="glorot_uniform"),
    tf.keras.layers.Dense(10, activation="softmax")
]

all_layers = preprocess_layers + nn_layers

In [21]:
def init_preprocess(sample_images, preprocess_layers):
    data = sample_images
    for preprocess_layer in preprocess_layers:
        data = preprocess_layer.adapt(sample_images)
    return preprocess_layers

def init_model(preprocess_layers, model_layers):
    model = tf.keras.Sequential(preprocess_layers + model_layers)
    model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
    return model

In [29]:
log_dir = "/log/tb"
tensorboard_cb = tf.keras.callbacks.TensorBoard(log_dir, update_freq=300)

sample_images = train_dataset.take(100).map(lambda image, label: image)
sample_images = np.concatenate(list(sample_images.as_numpy_iterator()), axis=0).astype(np.float32)
preprocess_layers = init_preprocess(sample_images, preprocess_layers)
model = init_model(preprocess_layers, nn_layers)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10,
    callbacks=[tensorboard_cb],
    verbose=2
)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1500/1500 - 23s - 15ms/step - accuracy: 0.8389 - loss: 0.4477 - val_accuracy: 0.8830 - val_loss: 0.3276
Epoch 2/10
1500/1500 - 16s - 10ms/step - accuracy: 0.8959 - loss: 0.2875 - val_accuracy: 0.8975 - val_loss: 0.3050
Epoch 3/10
1500/1500 - 15s - 10ms/step - accuracy: 0.9148 - loss: 0.2339 - val_accuracy: 0.8969 - val_loss: 0.3121
Epoch 4/10
1500/1500 - 15s - 10ms/step - accuracy: 0.9275 - loss: 0.1969 - val_accuracy: 0.9021 - val_loss: 0.3158
Epoch 5/10
1500/1500 - 15s - 10ms/step - accuracy: 0.9402 - loss: 0.1632 - val_accuracy: 0.8989 - val_loss: 0.3320
Epoch 6/10
1500/1500 - 15s - 10ms/step - accuracy: 0.9472 - loss: 0.1410 - val_accuracy: 0.9116 - val_loss: 0.3299
Epoch 7/10
1500/1500 - 52s - 35ms/step - accuracy: 0.9516 - loss: 0.1489 - val_accuracy: 0.9032 - val_loss: 0.3792
Epoch 8/10
1500/1500 - 15s - 10ms/step - accuracy: 0.9569 - loss: 0.1227 - val_accuracy: 0.9078 - val_loss: 0.3824
Epoch 9/10
1500/1500 - 15s - 10ms/step - accuracy: 0.9642 - loss: 0.1031 - val_accuracy: 0.